# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice:Signal analysis and ranking comparison. I chose signal analysis because my goal is to rank content pages by review priority using search performance signals such as impressions, clicks, CTR and average position. This approach fits my lane because it focuses directly on identifying which pages should be reviewed first. I will compare the new ranking with my week 4 baseline on the same validation period and using the same metric.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: time-aware split . I will train the model on earlier March dates and validate it on later March dates. This is more honest than a random split because the model should learn from past data and be evaluated on data that comes later. The same validation period will also be used to evaluate the week 4 baseline so the comparison is fair.

In [2]:
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
    repo_id="Flyrank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(march_path)

print("Rows loaded:", len(df))
print("date range:", df["report_date"].min(), "to", df["report_date"].max())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Rows loaded: 9841378
date range: 2026-03-01 to 2026-03-31


In [3]:
df["report_date"] = pd.to_datetime(df["report_date"])
train_df = df[df["report_date"] <= "2026-03-24"].copy()

val_df = df[df["report_date"] > "2026-03-24"].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Train dates:", train_df["report_date"].min(), "to", train_df["report_date"].max())
print("Validation dates", val_df["report_date"].min(), "to", val_df["report_date"].max())


Train rows: 7548489
Validation rows: 2292889
Train dates: 2026-03-01 00:00:00 to 2026-03-24 00:00:00
Validation dates 2026-03-25 00:00:00 to 2026-03-31 00:00:00


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
val_summary = (
    val_df
    .groupby("content_hash_id", as_index=False)
    .agg({
        "gsc_impressions": "sum",
        "gsc_clicks": "sum",
        "gsc_avg_position": "mean"
    })
)

val_summary["ctr"] = (
    val_summary["gsc_clicks"] /
    val_summary["gsc_impressions"].replace(0, pd.NA)
)

print(val_summary.head())
print("Rows", len(val_summary))

            content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position  \
0  content_000005d4ced12088               30           0         76.166667   
1  content_00001e488b74b799                0           0               NaN   
2  content_00007bd2985b77c3               14           0          3.833333   
3  content_00008950670cb6b5                0           0               NaN   
4  content_0000a348850eb1fc                0           0               NaN   

    ctr  
0   0.0  
1  <NA>  
2   0.0  
3  <NA>  
4  <NA>  
Rows 331436


In [13]:
ranking_df = val_summary.copy()

ranking_df["ctr"] = ranking_df["ctr"].fillna(0)
ranking_df["gsc_avg_position"] = ranking_df["gsc_avg_position"].fillna(100)

ranking_df["priority_score"] = (
    ranking_df["gsc_impressions"] *
    (1 - ranking_df["ctr"])  *
    ranking_df["gsc_avg_position"]
)

ranking_df = ranking_df.sort_values(
    "priority_score",
    ascending=False
)

print(ranking_df.head(10))

                 content_hash_id  gsc_impressions  gsc_clicks  \
149441  content_73aa61dcedebbf30            27069           2   
70657   content_36e53e9c707674fc            39542          58   
131959  content_66288edeb93b7c4f            79987         422   
131885  content_661a7734f691bef5            39549          11   
169222  content_82e35c4845e6c391            34717          19   
82065   content_3f9e8f387f3fe7e7            19619          12   
324200  content_fa84f5976d5fe3c1            20912          29   
211749  content_a3a1317f7c2bc3dd            21995           1   
221913  content_ab91e088440ace78            18444           0   
25144   content_136c4bf04b07b778            16130          12   

        gsc_avg_position       ctr  priority_score  
149441         49.217629  0.000074    1.332174e+06  
70657          31.773706  0.001467    1.254553e+06  
131959         13.738387  0.005276    1.093095e+06  
131885         26.531121  0.000278    1.048987e+06  
169222         30.1

/tmp/ipykernel_3019/4030547879.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ranking_df["ctr"] = ranking_df["ctr"].fillna(0)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.